# Patients Data Exploration
Synthetic Data prepared from `https://mitre.box.com/shared/static/aw9po06ypfb9hrau4jamtvtz0e5ziucz.zip`

In [2]:
import pandas as pd
import numpy as np
import re
import os
from PatientUtils import EDA_utils as eda
from PatientUtils import visualization_utils as vsz
%cd -q ..
%run ./local_settings.py # no peeky, i will create a similar thing thru `activate_setup.sh`
%cd -q Notebooks

EDA = eda.EDA()

In [3]:
# extract the files from the directory
pdata_dir = rf'{os.environ['PATIENT_DATA_DIR']}/PatientData'
csv_files = []
for file in os.listdir(pdata_dir):
    # only grab the CSV
    # if str(os.path.splitext(file)).lower() == 'csv':
    path = os.path.join(file)
    csv_files.append(path)
    continue

for name in csv_files:
    print(name)
    with open(fr'{pdata_dir}/{name}','r') as file:
        print(file.readline())
    print(50*'-')
    print()


devices.csv
START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,UDI

--------------------------------------------------

claims_transactions.csv
ID,CLAIMID,CHARGEID,PATIENTID,TYPE,AMOUNT,METHOD,FROMDATE,TODATE,PLACEOFSERVICE,PROCEDURECODE,MODIFIER1,MODIFIER2,DIAGNOSISREF1,DIAGNOSISREF2,DIAGNOSISREF3,DIAGNOSISREF4,UNITS,DEPARTMENTID,NOTES,UNITAMOUNT,TRANSFEROUTID,TRANSFERTYPE,PAYMENTS,ADJUSTMENTS,TRANSFERS,OUTSTANDING,APPOINTMENTID,LINENOTE,PATIENTINSURANCEID,FEESCHEDULEID,PROVIDERID,SUPERVISINGPROVIDERID

--------------------------------------------------

supplies.csv
DATE,PATIENT,ENCOUNTER,CODE,DESCRIPTION,QUANTITY

--------------------------------------------------

claims.csv
Id,PATIENTID,PROVIDERID,PRIMARYPATIENTINSURANCEID,SECONDARYPATIENTINSURANCEID,DEPARTMENTID,PATIENTDEPARTMENTID,DIAGNOSIS1,DIAGNOSIS2,DIAGNOSIS3,DIAGNOSIS4,DIAGNOSIS5,DIAGNOSIS6,DIAGNOSIS7,DIAGNOSIS8,REFERRINGPROVIDERID,APPOINTMENTID,CURRENTILLNESSDATE,SERVICEDATE,SUPERVISINGPROVIDERID,STATUS1,STATUS2,STATUSP,OUTSTAN

## Tables
### **Helper Functions**
For further insight into these functions, refer to `src/PatientData/EDA_utils.py`

### `basicDescriptiveColumnEDA()`
This will print the basic descriptive statistics and this will print the column distinct values if the thresholding set is met. This can either be an `INT` or `PERCENT`.

### `extractFromParentheses()`
In many of the columns (specifically with a `description` like column name) have extractable data from the parenthesis set. This will display a dataframe display of the unique values from the main `description`/other.

This will also include options to output mapping tables.

#### `GSIExtract()`
Pull out the GSI ID string info

### EDA
#### `devices.csv`

In [82]:
filename = 'devices.csv'

dev_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})

EDA.displayUniqueStringDataFromParentheses(df=dev_df)



        Column: START
        --------------------------------------------------
        
count                       89
unique                      87
top       2021-01-09T18:42:42Z
freq                         2
Name: START, dtype: object



        Column: STOP
        --------------------------------------------------
        
count                       14
unique                      14
top       2021-01-20T05:41:03Z
freq                         1
Name: STOP, dtype: object
<StringArray>
[                   nan, '2021-01-20T05:41:03Z', '2021-01-09T19:03:24Z',
 '2021-01-09T23:35:03Z', '2021-01-11T01:58:03Z', '2021-01-12T04:04:03Z',
 '2021-01-13T06:33:03Z', '2021-01-14T10:28:03Z', '2021-01-15T12:33:03Z',
 '2021-01-16T15:25:03Z', '2021-01-17T18:04:03Z', '2021-01-18T20:13:03Z',
 '2021-01-19T23:23:03Z', '2021-01-15T03:13:57Z', '2021-01-06T01:13:51Z']
Length: 15, dtype: str



        Column: PATIENT
        --------------------------------------------------
        
count              

In [77]:

### Print out a clear visual, just want to see if the 01, 11, 17, 10, 21 are universal to the DSet
pattern = r"\((\d{2})\)([^\(]+)"

for gsi in dev_df["UDI"].dropna().head(50):
    parts = dict(re.findall(pattern, gsi))

    print(f'''
    GTIN ('01'): {parts.get('01')}
    ProdDate ('11'): {parts.get('11')}
    ExpDate ('17'): {parts.get('17')}
    lot ('10'): {parts.get('10')}
    serial ('21'): {parts.get('21')}
    ''')

### Extract these into a df and put in the mapping tables
#pattern = r"\(01\)(?P<gtin_01>\d{14})\(11\)(?P<prod_date_11>\d{6})\(17\)(?P<exp_date_17>\d{6})\(10\)(?P<lot_10>\d+)\(21\)(?P<serial_21>\d+)"
gsi_extract = eda.GSIExtract(dev_df,'UDI')
gsi_extract.to_csv(f'{pdata_dir}/MapTables/UDI_GSI_Breakdown')

              gtin prod_date exp_date                  lot  \
0   32224845419831    170904   420919            707911340   
1   84789768157847    870830   120913  3106560426982724622   
2   87802513365737    150621   400705         286435681772   
3   71493530635894    830627   080711               697080   
4   60616412587042    080314   330329  1877073077696890188   
..             ...       ...      ...                  ...   
84  89715011359638    031117   281201          38238102401   
85  22711596160291    980808   230823        4298074405474   
86  65892347614918    210426   460511        2418453102475   
87  34382442192159    100917   351002         347951028605   
88  33501576283277    970815   220830    59669302583549844   

                 serial  
0               5755382  
1        38557448172153  
2               1981302  
3                274503  
4          338766942724  
..                  ...  
84  3684901247193991356  
85      654247834231910  
86          724545056

#### `claims_transactions.csv`

In [63]:
filename = 'claims_transactions.csv'

claimtrans_df = EDA.basicDescriptiveColumnEDA(filename)
EDA.displayUniqueStringDataFromParentheses(df=claimtrans_df)



        Column: ID
        --------------------------------------------------
        
count                                   711238
unique                                  711238
top       5c58de0b-3ba9-401d-2ce7-a4785ff35c15
freq                                         1
Name: ID, dtype: object



        Column: CLAIMID
        --------------------------------------------------
        
count                                   711238
unique                                  117889
top       8dcebf32-ed37-b68e-3405-a0cb20fe538d
freq                                       130
Name: CLAIMID, dtype: object
<StringArray>
['e413105d-6f23-34ef-724a-b42adab9df22',
 '8e430d76-6628-c3ac-8950-2acabeb34f86',
 '3a69d5f0-26d8-fe66-4a1b-1a7c9ab618c8',
 'af33009b-e02a-6959-e5cc-5dffc7398cbd',
 '3153e655-a843-2bf1-72eb-1f06a52e524a',
 '3d8c5921-885e-740d-8a14-3ee3e1118df7',
 '23b8db87-5ad9-387e-879e-4c2ebba99ab0',
 '988099b8-dad2-f862-35cb-c76cfffc046f',
 '350c3a2c-9262-c79b-1920-71250474a0e1',
 'fb2

/tmp/ipykernel_4377/2881721148.py:22: RuntimeWarning: invalid value encountered in scalar divide
  if dislength/length <= threshold['PERCENT']:
/tmp/ipykernel_4377/2881721148.py:22: RuntimeWarning: invalid value encountered in scalar divide
  if dislength/length <= threshold['PERCENT']:


<StringArray>
[                                     'Well child visit (procedure)',
                                    'Hep B  adolescent or pediatric',
                             'Medication Reconciliation (procedure)',
                                                     'Hib (PRP-OMP)',
                                             'rotavirus  monovalent',
                                                               'IPV',
                                                              'DTaP',
                                     'Pneumococcal conjugate PCV 13',
                'Influenza  seasonal  injectable  preservative free',
                                             'Encounter for problem',
 ...
                'piperacillin 4000 MG / tazobactam 500 MG Injection',
                             '4 ML Norepinephrine 1 MG/ML Injection',
                        '1 ML Vasopressin (USP) 20 UNT/ML Injection',
                                     'Terfenadine 60 MG Oral Tablet',
 

/tmp/ipykernel_4377/2881721148.py:22: RuntimeWarning: invalid value encountered in scalar divide
  if dislength/length <= threshold['PERCENT']:
/tmp/ipykernel_4377/2881721148.py:22: RuntimeWarning: invalid value encountered in scalar divide
  if dislength/length <= threshold['PERCENT']:


<StringArray>
['748f8357-6cc7-551d-f31a-32fa2cf84126',
 '5a4735ae-423f-6563-28ab-b3d11b49b2d4',
 '0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3',
 '6e93bcf9-45a4-8528-0120-1c1eaa930faf',
 '8b6787c3-4316-a0cb-899d-4746525c319f',
 '8f424287-ee3a-c144-bc1d-3ba926e93fd5',
 'fb15e123-fea7-cae8-6d49-ee9d2a85fc84',
 '01efcc52-15d6-51e9-faa2-bee069fcbe44',
 '1a7debfc-9582-7f23-a109-4f154a182ee2',
 'bf38c711-941f-7509-f9ec-b864d6929f3f',
 ...
 'dc20e0f3-5f9b-0008-ae40-29ed26193987',
 'a7b3614b-3840-6d16-a23d-37f2f73938cd',
 '794e7686-7af0-50ae-3d4a-70e5db2eee70',
 'd135bbff-a4f5-653b-5462-68af428138be',
 '090096d3-9404-3cb5-d1eb-583bf4c39180',
 '230e2215-38ab-9371-842d-a44d27ae4090',
 'db101ad8-66e2-9feb-e0cf-b2618f873c3a',
 '1516d2e6-4846-5f1e-fe27-c1ebb9a39f72',
 'b2a4d90b-a2f5-1c88-0fb6-ba49b1487d37',
 '0732798b-fb31-183b-22b6-efb205ee6502']
Length: 61459, dtype: str



        Column: LINENOTE
        --------------------------------------------------
        
count    0.0
mean     NaN
std      NaN


#### `supplies.csv`

In [59]:
filename = 'supplies.csv'

supplies_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})
EDA.displayUniqueStringDataFromParentheses(df=supplies_df)



        Column: DATE
        --------------------------------------------------
        
count           1573
unique           158
top       2021-01-09
freq              38
Name: DATE, dtype: object



        Column: PATIENT
        --------------------------------------------------
        
count                                     1573
unique                                      20
top       78da7c78-d491-32b2-7ea2-aebb2517d27e
freq                                       148
Name: PATIENT, dtype: object
<StringArray>
['8fa5a097-1970-9370-4193-a7baa3d235b5',
 '41ba25b8-f5ca-3bce-c26f-64b5ce13e525',
 'e8ffd460-0685-7966-5e43-d24315e2f40c',
 '78da7c78-d491-32b2-7ea2-aebb2517d27e',
 '86dbfa87-dc1c-e529-f39b-4910e4deb3fb',
 '319372b8-27b1-b827-5c89-ac991fc17407',
 'aa2f41c3-412f-6635-26df-de6f46013766',
 'd66b5418-06cb-fc8a-8c13-85685b6ac939',
 'd868e198-8ebe-2338-5691-4c1de3421c65',
 '76b289fd-e825-734c-8446-316f59643593',
 '20a75f26-b297-4120-72e6-602ee5e9f4e4',
 '050358a1-fcf7-1182-7a

#### `Description`
```txt
[                       'Disposable air-purifying respirator (physical object)',
 'Nitrile examination/treatment glove  non-powdered  sterile (physical object)',
                                 'Isolation gown  single-use (physical object)',
                                                'Face shield (physical object)',
                                             'Alcohol disinfectant (substance)',
                                       'Antiseptic towelette (physical object)',
                                   'Isolation gown  reusable (physical object)',
                            'Operating room gown  single-use (physical object)',
                                   'Surgical cap  single-use (physical object)',
                                 'Protective glasses  device (physical object)',
                             'Laryngoscope blade  single-use (physical object)',
                        'Basic endotracheal tube  single-use (physical object)',
                       'Endotracheal tube stylet  single-use (physical object)',
                                            'Syringe  device (physical object)',
                                             'Suction system (physical object)',
                                                  'Lubricant (physical object)',
                                   'Endotracheal tube holder (physical object)',
                                               'Viral filter (physical object)',
                             'Carbon dioxide breath analyzer (physical object)',
                                   'Nasogastric tube  device (physical object)']
```

Split this into two columns (snowflaked): `supply_description` ,`physobj_or_subst`

#### `claims.csv`

In [41]:
claims_df = EDA.basicDescriptiveColumnEDA('claims.csv',threshold={'INT':25})

eda.displayUniqueStringDataFromParentheses(df=claims_df)



        Column: Id
        --------------------------------------------------
        
count                                   117889
unique                                  117889
top       e413105d-6f23-34ef-724a-b42adab9df22
freq                                         1
Name: Id, dtype: object



        Column: PATIENTID
        --------------------------------------------------
        
count                                   117889
unique                                    1163
top       ef167059-cef0-12c4-49db-993ca3a20c01
freq                                      3551
Name: PATIENTID, dtype: object



        Column: PROVIDERID
        --------------------------------------------------
        
count                                   117889
unique                                    1123
top       43b6d00b-7643-36f8-8043-456d4eda39eb
freq                                      4262
Name: PROVIDERID, dtype: object



        Column: PRIMARYPATIENTINSURANCEID
        -------------

In [88]:
claims_df[['Diagnosis1'.upper(),'Diagnosis2'.upper(),'Diagnosis3'.upper(),'Diagnosis4'.upper()]].drop_duplicates().sample(25)

,DIAGNOSIS1,DIAGNOSIS2,DIAGNOSIS3,DIAGNOSIS4
24791,301011002,301011002.0,160903007.0,NaN
37760,126906006,126906006.0,92691004.0,1.609030e+08
35838,25064002,267102003.0,84229001.0,3.866610e+08
10967,87433001,160904001.0,73595000.0,NaN
54554,72892002,72892002.0,79586000.0,3.599901e+07
40871,363406005,160903007.0,NaN,NaN
12309,444814009,422650009.0,NaN,NaN
3754,49727002,43724002.0,386661006.0,3.695501e+07
26122,83664006,83664006.0,NaN,NaN
9670,59621000,224355006.0,224299000.0,7.068930e+08


#### `encounters.csv`

In [ ]:
encounters_df = EDA.basicDescriptiveColumnEDA('encounters.csv',threshold={'INT':100})

EDA.displayUniqueStringDataFromParentheses(df=encounters_df)



        Column: Id
        --------------------------------------------------
        
count                                    61459
unique                                   61459
top       748f8357-6cc7-551d-f31a-32fa2cf84126
freq                                         1
Name: Id, dtype: object



        Column: START
        --------------------------------------------------
        
count                    61459
unique                   58419
top       1931-08-04T21:04:17Z
freq                        11
Name: START, dtype: object



        Column: STOP
        --------------------------------------------------
        
count                    61459
unique                   58805
top       1931-08-04T21:19:17Z
freq                        11
Name: STOP, dtype: object



        Column: PATIENT
        --------------------------------------------------
        
count                                    61459
unique                                    1163
top       ef167059-cef0-1

#### `DESCRIPTION`
Again split this column into two, `ENCOUNTER_DESCRIPTION`, `ENCOUNTER_DESC_DETAIL`. Here it appears `procedure`,`qualifer offer`, and NULL will be the expected value for `ENCOUNTER_DESC_DETAIL`

```txt
[                                          'Well child visit (procedure)',
                                                  'Encounter for problem',
                                                  'Encounter for symptom',
                                         'Urgent care clinic (procedure)',
                                   'Emergency room admission (procedure)',
                                               'Encounter for 'check-up'',
                                             'Consultation for treatment',
                                            'Patient encounter procedure',
                             'General examination of patient (procedure)',
                                       'Hypertension follow-up encounter',
                                     'Encounter for check up (procedure)',
       'Administration of vaccine to produce active immunity (procedure)',
                                                 'Prenatal initial visit',
                                                         'Prenatal visit',
                                 'Obstetric emergency hospital admission',
                                                        'Postnatal visit',
                                            'Patient-initiated encounter',
                                      'Encounter for symptom (procedure)',
                                               'Emergency Room Admission',
                                      'Encounter for problem (procedure)',
                                'Patient encounter procedure (procedure)',
                                       'Admission to surgical department',
                                                    'Death Certification',
                                                    'Follow-up encounter',
                                        'Follow-up encounter (procedure)',
                           'Hospital admission for isolation (procedure)',
                           'Admission to intensive care unit (procedure)',
                        'Hospital admission  for observation (procedure)',
                                                       'Asthma follow-up',
                                'Emergency hospital admission for asthma',
                                   'Allergic disorder initial assessment',
                                                   'Outpatient procedure',
                                 'Allergic disorder follow-up assessment',
                                                    'Encounter Inpatient',
 'Periodic reevaluation and management of healthy individual (procedure)',
                                 'Discussion about treatment (procedure)',
                              'Postoperative follow-up visit (procedure)',
                                'Screening surveillance (regime/therapy)',
                                                                 'Stroke',
                                                     'Hospital admission',
                                                    'Emergency Encounter',
                                                         'Cardiac Arrest',
                                 'Drug rehabilitation and detoxification',
                                            'Follow-up visit (procedure)',
             'Domiciliary or rest home patient evaluation and management',
                                                  'Encounter for Problem',
                                        'Non-urgent orthopedic admission',
                                                  'Myocardial Infarction',
                               'Admission to thoracic surgery department',
  'Physician visit with evaluation AND/OR management service (procedure)',
            'Initial Psychiatric Interview with mental status evaluation',
                                          'posttraumatic stress disorder',
                                        'Telephone encounter (procedure)',
                                                           'Office Visit',
                                 'Telemedicine consultation with patient',
                                   'Gynecology service (qualifier value)',
            'Diagnosis of cystic fibrosis using sweat test and gene test',
                                                 'Encounter for check up',
                                                  'Inpatient stay 3 days']
```

#### `careplans.csv`

In [ ]:
filename = 'careplans.csv'

careplans_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':50})

EDA.displayUniqueStringDataFromParentheses(df=careplans_df)



        Column: Id
        --------------------------------------------------
        
count                                     3931
unique                                    3931
top       6d10e8ad-cdf8-db60-ff71-688eae2861c2
freq                                         1
Name: Id, dtype: object



        Column: START
        --------------------------------------------------
        
count           3931
unique          3069
top       2014-02-01
freq              20
Name: START, dtype: object



        Column: STOP
        --------------------------------------------------
        
count           2252
unique          1758
top       2014-03-08
freq              16
Name: STOP, dtype: object



        Column: PATIENT
        --------------------------------------------------
        
count                                     3931
unique                                    1068
top       e0807fab-552e-931f-366f-761dff93a52e
freq                                        16
Name: PATIE

Same deal with `DESCRIPTION` here, the values for detail will be:
* `Procedure`
* `record artifact`
* `regime/therapy`
* NULL

#### `observations.csv`

In [46]:
filename = 'observations.csv'

obs_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})

EDA.displayUniqueStringDataFromParentheses(df=obs_df)



        Column: DATE
        --------------------------------------------------
        
count                   531144
unique                   63811
top       2001-01-09T21:04:17Z
freq                       108
Name: DATE, dtype: object



        Column: PATIENT
        --------------------------------------------------
        
count                                   531144
unique                                    1163
top       ef167059-cef0-12c4-49db-993ca3a20c01
freq                                      9297
Name: PATIENT, dtype: object



        Column: ENCOUNTER
        --------------------------------------------------
        
count                                   499482
unique                                   21025
top       77c7ccff-dbc5-2ee2-7d48-52f8af2eeb67
freq                                       678
Name: ENCOUNTER, dtype: object



        Column: CATEGORY
        --------------------------------------------------
        
count     499482
unique         8
to

Looks like for all of them, the descriptions will have some level of additional column expansion. The nice thing is the `detail` columns I have been creating have been sourced from `()`, making these easy pickings for regex. This column does bring up a serious question: is there any further bucketing we need. There might be further generalizations we can use to create easier navigable. So maybe here with observations, there could be a way to classify the actions into:
- Types of actions
- Mood notes
- Other

#### `payers.csv`

In [47]:
filename = 'payers.csv'

payers_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})

EDA.displayUniqueStringDataFromParentheses(df=payers_df)



        Column: Id
        --------------------------------------------------
        
count                                       10
unique                                      10
top       b3221cfc-24fb-339e-823d-bc4136cbc4ed
freq                                         1
Name: Id, dtype: object
<StringArray>
['b3221cfc-24fb-339e-823d-bc4136cbc4ed',
 '7caa7254-5050-3b5e-9eae-bd5ea30e809c',
 '7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a',
 'd47b3510-2895-3b70-9897-342d681c769d',
 '6e2f1a2d-27bd-3701-8d08-dae202c58632',
 '5059a55e-5d6e-34d1-b6cb-d83d16e57bcf',
 '4d71f845-a6a9-3c39-b242-14d25ef86a8d',
 '047f6ec3-6215-35eb-9608-f9dda363a44c',
 '42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a',
 'b1c428d6-4f07-31e0-90f0-68ffa6ff8c76']
Length: 10, dtype: str



        Column: NAME
        --------------------------------------------------
        
count                10
unique               10
top       Dual Eligible
freq                  1
Name: NAME, dtype: object
<StringArray>
[         'Dual Eligible'

So we will need to actually get in the weeds here, but this should be a primary part of the analysis for the 'value opportunities' for both raw amounts and willingness to cover

#### `imaging_studies.csv`

In [48]:
filename = 'imaging_studies.csv'

imaging_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})

EDA.displayUniqueStringDataFromParentheses(df=imaging_df)



        Column: Id
        --------------------------------------------------
        
count                                   151637
unique                                     920
top       63944f08-0fa2-3ae8-259d-19351104ef5c
freq                                       500
Name: Id, dtype: object



        Column: DATE
        --------------------------------------------------
        
count                   151637
unique                     920
top       1992-05-19T23:37:28Z
freq                       500
Name: DATE, dtype: object



        Column: PATIENT
        --------------------------------------------------
        
count                                   151637
unique                                     238
top       4b0301f7-6c9e-e170-d540-eda9466a29d7
freq                                     27816
Name: PATIENT, dtype: object



        Column: ENCOUNTER
        --------------------------------------------------
        
count                                   151637
un

`BODYSITE_DESCRIPTION` and `SOP_DESCRIPTION` need to be split into more columns. SOP will just essentially be the equivalent of a display filter so only the `for presentation` one is picked up for display IF patient visible. Body site is has `structure` seemingly seperating the difference between bone vs organ or muscle. Use this to set `soft tissue` vs `bone` for `bodysite_structure_description`

#### `immunizations.csv`

In [49]:
filename = 'immunizations.csv'

immun_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})

EDA.displayUniqueStringDataFromParentheses(df=immun_df)



        Column: DATE
        --------------------------------------------------
        
count                    17009
unique                   11618
top       2012-05-04T02:50:03Z
freq                         7
Name: DATE, dtype: object



        Column: PATIENT
        --------------------------------------------------
        
count                                    17009
unique                                    1158
top       ed01aca0-b28f-4b31-711b-3114e5d3c661
freq                                        36
Name: PATIENT, dtype: object



        Column: ENCOUNTER
        --------------------------------------------------
        
count                                    17009
unique                                   11796
top       0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3
freq                                         5
Name: ENCOUNTER, dtype: object



        Column: CODE
        --------------------------------------------------
        
count    17009.000000
mean       118.944

#### `payer_transitions.csv`

In [50]:
filename = 'payer_transitions.csv'

payertrans_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})

EDA.displayUniqueStringDataFromParentheses(df=payertrans_df)




        Column: PATIENT
        --------------------------------------------------
        
count                                    53101
unique                                    1163
top       e23c7a6e-6ef2-3932-1361-b73b4c8a3961
freq                                       255
Name: PATIENT, dtype: object



        Column: MEMBERID
        --------------------------------------------------
        
count                                    43555
unique                                   32119
top       7f70dd21-d310-77fb-f341-41f0a8a0fe53
freq                                        47
Name: MEMBERID, dtype: object



        Column: START_YEAR
        --------------------------------------------------
        
count                    53101
unique                   47005
top       1913-06-17T21:04:17Z
freq                        11
Name: START_YEAR, dtype: object



        Column: END_YEAR
        --------------------------------------------------
        
count                    5

#### `conditions.csv`

In [51]:
filename = 'conditions.csv'

conditions_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})

EDA.displayUniqueStringDataFromParentheses(df=conditions_df)



        Column: START
        --------------------------------------------------
        
count          38094
unique         12995
top       2014-02-03
freq              54
Name: START, dtype: object



        Column: STOP
        --------------------------------------------------
        
count          29925
unique         11508
top       2021-03-25
freq              25
Name: STOP, dtype: object



        Column: PATIENT
        --------------------------------------------------
        
count                                    38094
unique                                    1147
top       e23c7a6e-6ef2-3932-1361-b73b4c8a3961
freq                                       460
Name: PATIENT, dtype: object



        Column: ENCOUNTER
        --------------------------------------------------
        
count                                    38094
unique                                   26904
top       7044bdaf-bcf5-af38-545b-681150095994
freq                                        13

Again, another `DESCRIPTION` column

#### `organizations.csv`

In [52]:
filename = 'organizations.csv'

organizations_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})
EDA.displayUniqueStringDataFromParentheses(df=organizations_df)



        Column: Id
        --------------------------------------------------
        
count                                     1127
unique                                    1127
top       ef58ea08-d883-3957-8300-150554edc8fb
freq                                         1
Name: Id, dtype: object



        Column: NAME
        --------------------------------------------------
        
count                           1127
unique                           984
top       STEWARD MEDICAL GROUP  INC
freq                              22
Name: NAME, dtype: object



        Column: ADDRESS
        --------------------------------------------------
        
count                1127
unique               1016
top       100 HIGHLAND ST
freq                    8
Name: ADDRESS, dtype: object



        Column: CITY
        --------------------------------------------------
        
count          1127
unique          319
top       BROOKLINE
freq             36
Name: CITY, dtype: object



     

#### `procedures.csv`

In [53]:
filename = 'procedures.csv'

proc_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})
EDA.displayUniqueStringDataFromParentheses(df=proc_df)


        Column: START
        --------------------------------------------------
        
count                    83823
unique                   64956
top       2014-08-16T03:25:13Z
freq                        23
Name: START, dtype: object



        Column: STOP
        --------------------------------------------------
        
count                    83823
unique                   70885
top       2014-08-16T03:40:13Z
freq                        22
Name: STOP, dtype: object



        Column: PATIENT
        --------------------------------------------------
        
count                                    83823
unique                                    1162
top       ef167059-cef0-12c4-49db-993ca3a20c01
freq                                      1532
Name: PATIENT, dtype: object



        Column: ENCOUNTER
        --------------------------------------------------
        
count                                    83823
unique                                   25854
top       b63

Another description column. Will look at this more in depth to see if there is any further detail like the others.

#### `providers.csv`

In [54]:
filename = 'providers.csv'

providers_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})
EDA.displayUniqueStringDataFromParentheses(df=providers_df)


        Column: Id
        --------------------------------------------------
        
count                                     5056
unique                                    5056
top       c23e8780-6030-37ec-8d02-8c6e3def10ac
freq                                         1
Name: Id, dtype: object



        Column: ORGANIZATION
        --------------------------------------------------
        
count                                     5056
unique                                    1127
top       6d59a1fd-21d9-3966-9df2-49161ca3667d
freq                                       345
Name: ORGANIZATION, dtype: object



        Column: NAME
        --------------------------------------------------
        
count                      5056
unique                     3735
top       Foster87 Abernathy524
freq                         16
Name: NAME, dtype: object



        Column: GENDER
        --------------------------------------------------
        
count     5056
unique       2
top     

#### `patients.csv`

In [55]:
filename = 'patients.csv'

patients_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})
EDA.displayUniqueStringDataFromParentheses(df=patients_df)


        Column: Id
        --------------------------------------------------
        
count                                     1163
unique                                    1163
top       b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85
freq                                         1
Name: Id, dtype: object



        Column: BIRTHDATE
        --------------------------------------------------
        
count           1163
unique           987
top       1913-06-10
freq              11
Name: BIRTHDATE, dtype: object



        Column: DEATHDATE
        --------------------------------------------------
        
count            163
unique           163
top       2009-11-13
freq               1
Name: DEATHDATE, dtype: object



        Column: SSN
        --------------------------------------------------
        
count            1163
unique           1163
top       999-65-3251
freq                1
Name: SSN, dtype: object



        Column: DRIVERS
        -------------------------------------

#### `allergies.csv`

In [56]:
filename = 'allergies.csv'

allergies_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})

EDA.displayUniqueStringDataFromParentheses(df=allergies_df)


        Column: START
        --------------------------------------------------
        
count            794
unique           179
top       1993-05-27
freq              12
Name: START, dtype: object



        Column: STOP
        --------------------------------------------------
        
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: STOP, dtype: float64
[nan]



        Column: PATIENT
        --------------------------------------------------
        
count                                      794
unique                                     179
top       20a75f26-b297-4120-72e6-602ee5e9f4e4
freq                                        12
Name: PATIENT, dtype: object



        Column: ENCOUNTER
        --------------------------------------------------
        
count                                      794
unique                                     179
top       9aed9771-89ce-daa6-d81d-2c70e27410c8
freq               

This has 3 `Description` columns, with multiple `()` extractable info

#### `medications.csv`

In [57]:
filename = 'medications.csv'

medi_df = EDA.basicDescriptiveColumnEDA(filename,threshold={'INT':25})

EDA.displayUniqueStringDataFromParentheses(df=medi_df)



        Column: START
        --------------------------------------------------
        
count                    56430
unique                   25749
top       1997-12-23T21:04:17Z
freq                        29
Name: START, dtype: object



        Column: STOP
        --------------------------------------------------
        
count                    53717
unique                   23895
top       1997-12-23T21:04:17Z
freq                        26
Name: STOP, dtype: object



        Column: PATIENT
        --------------------------------------------------
        
count                                    56430
unique                                    1117
top       e23c7a6e-6ef2-3932-1361-b73b4c8a3961
freq                                      2117
Name: PATIENT, dtype: object



        Column: PAYER
        --------------------------------------------------
        
count                                    56430
unique                                      10
top       b1c428d

### Looking at distributions
The wonderful thing about data is that there is so many patterns and features that can correlate to positive results, the negative is that we must seperate correlation and cause. While there are certainly so verbose analyses we can do here, let's flex some SME knowledge to incorporate:

1. Patient visit trends
2. Patient pay trends
3. HCP Coverage & Treatments
    - Claims oriented
4. HCP $ correlations
5. Organization Coverage & Treatments
    - Claims oriented
6. Organization $ correlations
    - Potentially see if anything when joing the affiliated HCP to organizatiosn
7. Equipment / Devices / Careplan Detail Analysis

From here, we might see the picture more clearly on where to explore next and where to make hay.

In [ ]:
from PatientUtils import Bayesian_utils as bayes
from PatientUtils import ML_utils as ml

#### 1. Patient Visit Trends

`REASONDESCRIPTION` needs to be looked into more for additional splitting

### Notes on Data Models